# Prueba a ciegas de verdad: ¿WSI encuentra el arreglo del loop sin saber la respuesta?

Las corridas anteriores tenían reglas que apuntaban a p50c (solo capas de atención completa, rango de origen, alpha 6 y 64 neuronas por defecto). Aquí **no hay ninguna**:

- **Cualquiera de las 24 capas.** Se ordenan solo por cuánto separan loop de normal (d' medido en datos apartados).
- **Rejilla fija** congelada en el código: 1, 2 o 3 capas × 16/64/256/todas las neuronas × alpha 2/4/6/8/12 = **60 arreglos**.
- **Ronda 1:** los 60 en 32 problemas de *train* a 16k. **Ronda 2:** los 6 mejores en 64 problemas *train* nuevos. Gana el mejor de la ronda 2.
- **p50c no existe en el disco** durante el descubrimiento: el repo se clona sin `benchmarks/`. Se descarga solo después de `freeze`.
- **Test una sola vez** (200 GSM8K, 16k): base, ciego siempre activo, ciego vía MoI, control con dirección aleatoria (mismas capas, neuronas y alpha), y p50c.

**Uso:** `L4` → `Run all` → acepta Drive. Tarda **~4–5 h** (casi todo es la ronda 1). Si se corta, `Run all` otra vez: lo terminado se salta. El resultado queda en `MyDrive/wsi_runs/FINALES/discover_blind.json` y el runtime se apaga **solo si ese archivo existe**.

In [ ]:
W = "/content/work"
REPO = "/content/wsi"
S = f"{REPO}/scripts/discover_blind.py"
RUN_NAME = "discover_blind"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm
import vllm, transformers; print("vllm", vllm.__version__, "| transformers", transformers.__version__)

In [ ]:
# Progreso en Google Drive: lo que ya terminó no se repite aunque se reinicie el runtime
import os, glob, shutil, json, subprocess
from google.colab import drive
drive.mount("/content/drive")
DRIVE = f"/content/drive/MyDrive/wsi_runs/{RUN_NAME}"
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
os.makedirs(DRIVE, exist_ok=True); os.makedirs(FINALES, exist_ok=True); os.makedirs(W, exist_ok=True)
GUARDAR = ("*.json", "*.jsonl", "*.npz")

def guardar():
    for pat in GUARDAR:
        for f in glob.glob(f"{W}/{pat}"):
            shutil.copy2(f, DRIVE)

def restaurar():
    for pat in GUARDAR:
        for f in glob.glob(f"{DRIVE}/{pat}"):
            shutil.copy2(f, W)

def hecho(*archivos):
    return all(os.path.exists(f"{DRIVE}/{a}") for a in archivos)

print("ya guardado en Drive:", sorted(os.listdir(DRIVE)) or "nada todavía (empieza de cero)")

In [ ]:
# Código SIN p50c: se clonan solo scripts/ (benchmarks/, donde vive p50c, no se baja)
if not os.path.exists(REPO):
    !git clone -q --filter=blob:none --sparse https://github.com/krexito/weight-space-index.git {REPO}
    !cd {REPO} && git sparse-checkout set scripts
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
assert not os.path.exists(f"{REPO}/benchmarks"), "p50c no debe estar en el disco durante el descubrimiento"
print("código congelado en el commit", COMMIT)

In [ ]:
# 0. requests (GSM8K train para descubrir, test aparte) + modelo base
!python {S} prepare --work {W}
restaurar()

In [ ]:
# 1. el modelo base razona en 300 problemas de train (aquí aparecen los loops)
if not hecho("outputs_elicit.jsonl"):
    !python {S} gen --work {W} --model {W}/qwen-base --requests requests_elicit.jsonl --out outputs_elicit.jsonl
    guardar()

In [ ]:
# 2. diagnóstico en las 24 capas, sin filtros
if not hecho("diagnosis_blind.json"):
    !python {S} diagnose --work {W}
    guardar()
d = json.load(open(f"{W}/diagnosis_blind.json"))
print("capas ordenadas por d' (datos apartados):", d["layer_rank"])

In [ ]:
# 3. ronda 1: los 60 arreglos en dev1 (32 problemas train, 16k). Lo más largo (~3 h). Guarda en Drive tras cada uno.
!python {S} search --work {W} --round 1 --sync-dir {DRIVE}
guardar()

In [ ]:
# 4. ronda 2: los 6 mejores en dev2 (64 problemas train nuevos) -> elección
!python {S} search --work {W} --round 2 --sync-dir {DRIVE}
guardar()

In [ ]:
# 5. CONGELAR: la elección y el hash del código quedan fijos antes de tocar el test o p50c
!python {S} freeze --work {W} --git-commit {COMMIT}
guardar()
print(json.load(open(f"{W}/frozen.json")))

In [ ]:
# 6. test (200 problemas, 16k): base, ciego siempre activo, ciego vía MoI, control aleatorio
if not hecho("outputs_test_base.jsonl"):
    !python {S} gen --work {W} --model {W}/qwen-base --requests requests_test.jsonl --out outputs_test_base.jsonl
    guardar()
!python {S} plan-moi --work {W}
guardar()
for kind, fixed, moi in (("wsi", "outputs_test_blind.jsonl", "outputs_test_moi_blind.jsonl"),
                         ("random", "outputs_test_random.jsonl", "outputs_test_moi_random.jsonl")):
    if hecho(fixed, moi):
        continue
    CK = subprocess.check_output(["python", S, "build-chosen", "--work", W, "--direction", kind], text=True).strip().splitlines()[-1]
    if not hecho(fixed):
        !python {S} gen --work {W} --model {CK} --requests requests_test.jsonl --out {fixed}
        guardar()
    if not hecho(moi):
        !python {S} gen --work {W} --model {CK} --requests requests_test_moi.jsonl --out {moi}
        guardar()
    shutil.rmtree(CK, ignore_errors=True)

In [ ]:
# 7. examen: recién ahora se baja p50c
assert os.path.exists(f"{W}/frozen.json"), "sin freeze no hay examen"
!cd {REPO} && git sparse-checkout add benchmarks/exp8_math
if not os.path.exists(f"{W}/qwen-p50c"):
    !cd {REPO}/scripts && python -c "import moi_loop_router_vllm as lr; from pathlib import Path; print(lr.build_p50c(Path('{W}/qwen-base'), Path('{REPO}/benchmarks/exp8_math'), Path('{W}/qwen-p50c')))"
if not hecho("outputs_test_p50c.jsonl"):
    !python {S} gen --work {W} --model {W}/qwen-p50c --requests requests_test.jsonl --out outputs_test_p50c.jsonl
    guardar()
!python {S} report --work {W} --p50c-dir {W}/qwen-p50c
guardar()

In [ ]:
# 8. resultado final a Drive (FINALES): primero a un tmp, luego os.replace
import datetime
res = json.load(open(f"{W}/results_discover_blind.json"))
res["_final"] = {"complete": True, "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
                 "notebook": "discover_blind_colab.ipynb", "commit": COMMIT}
dst = f"{FINALES}/{RUN_NAME}.json"
tmp = dst + ".tmp"
with open(tmp, "w") as f:
    json.dump(res, f, indent=1)
os.replace(tmp, dst)
print("guardado en", dst)
try:
    from google.colab import files
    files.download(dst)
except Exception as e:
    print("descarga directa no disponible:", e)

In [ ]:
# Apagar el runtime: solo si el resultado final ya está en Drive
AUTO_APAGAR = True
import time
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst)).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    time.sleep(90)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")